# Data Cleaning Pipeline - Quy Hoạch Real Estate

## Mục Đích
Xây dựng một quy trình tiền xử lý dữ liệu (Data Cleaning) hoàn chỉnh cho tập dữ liệu bất động sản quy hoạch.
Đạt tiêu chuẩn: **Modular OOP**, **Data Assertions**, **Reproducibility**, **Clean Code (PEP 8)**, **End-to-End Pipeline**.

---

## 📋 Table of Contents
1. [Setup & Configuration](#setup)
2. [Load Data & EDA](#eda)
3. [Define Custom Transformers](#transformers)
4. [Execute Data Cleaning Pipeline](#pipeline)
5. [Train/Test Split & Export](#export)
6. [Final Data Assertions](#assertions)

---

# 1. Setup & Configuration <a name="setup"></a>

## 1.1 Import Libraries & Check Requirements

In [1]:
# ============================================================================
# CELL 1: Import Libraries & Print Requirements
# ============================================================================

import sys
import os
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from abc import ABC, abstractmethod
import warnings

import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import unicodedata
import logging

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# Print Python version and key library versions
print("="*80)
print("ENVIRONMENT & REQUIREMENTS")
print("="*80)
print(f"Python Version: {sys.version}")
print(f"\nKey Libraries:")
libraries = {
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'scikit-learn': __import__('sklearn').__version__,
    'matplotlib': plt.matplotlib.__version__,
    'seaborn': sns.__version__
}
for lib, version in libraries.items():
    print(f"  - {lib}: {version}")
print("="*80)
print()

ENVIRONMENT & REQUIREMENTS
Python Version: 3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]

Key Libraries:
  - pandas: 2.3.3
  - numpy: 2.2.5
  - scikit-learn: 1.7.1
  - matplotlib: 3.10.8
  - seaborn: 0.13.2



## 1.2 Configure Paths & Create Output Directory

In [5]:
# ============================================================================
# CELL 2: Configure Relative Paths & Create Output Directory
# ============================================================================

# Use relative paths for reproducibility
BASE_DIR = Path.cwd()
DATA_INPUT_PATH = BASE_DIR / "..\\data\\raw\\Final_Merged_2605_quyhoach.csv"
OUTPUT_DIR = BASE_DIR / "data_output"

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Define output file paths
TRAIN_OUTPUT_PATH = OUTPUT_DIR / "1_data_cleaned_train.csv"
TEST_OUTPUT_PATH = OUTPUT_DIR / "1_data_cleaned_test.csv"

print(f"📁 Base Directory: {BASE_DIR}")
print(f"📁 Input Data: {DATA_INPUT_PATH}")
print(f"📁 Output Directory: {OUTPUT_DIR}")
print(f"📁 Train Output: {TRAIN_OUTPUT_PATH}")
print(f"📁 Test Output: {TEST_OUTPUT_PATH}")
print()

# Verify input file exists
assert DATA_INPUT_PATH.exists(), f"Input file not found at {DATA_INPUT_PATH}"
print(f"✓ Input file exists and is accessible")
print()

📁 Base Directory: d:\DS108
📁 Input Data: d:\DS108\..\data\raw\Final_Merged_2605_quyhoach.csv
📁 Output Directory: d:\DS108\data_output
📁 Train Output: d:\DS108\data_output\1_data_cleaned_train.csv
📁 Test Output: d:\DS108\data_output\1_data_cleaned_test.csv



AssertionError: Input file not found at d:\DS108\..\data\raw\Final_Merged_2605_quyhoach.csv

---

# 2. Load Data & Exploratory Data Analysis (EDA) <a name="eda"></a>

## 2.1 Define DataAnalyzer Class

In [ ]:
# ============================================================================
# CELL 3: Define DataAnalyzer Class for Comprehensive EDA
# ============================================================================

class DataAnalyzer:
    """
    A comprehensive data analyzer for exploratory data analysis (EDA).
    
    This class provides methods to analyze dataset characteristics including:
    - Data shape and duplicates
    - Missing value patterns
    - Descriptive statistics (for numeric columns)
    - Categorical value distribution
    """
    
    def __init__(self, df: pd.DataFrame, name: str = "Dataset"):
        """
        Initialize the DataAnalyzer.
        
        Args:
            df (pd.DataFrame): The dataset to analyze
            name (str): Name of the dataset for display purposes
        """
        self.df = df
        self.name = name
    
    def comprehensive_eda(self) -> Dict[str, Any]:
        """
        Perform comprehensive exploratory data analysis and print detailed report.
        
        Returns:
            Dict[str, Any]: Dictionary containing all EDA metrics
        """
        print("\n" + "="*80)
        print(f"COMPREHENSIVE EDA - {self.name.upper()}")
        print("="*80 + "\n")
        
        eda_results = {}
        
        # 1. Dataset Shape
        print("📊 DATASET SHAPE")
        print("-" * 80)
        n_rows, n_cols = self.df.shape
        print(f"  Total Rows: {n_rows:,}")
        print(f"  Total Columns: {n_cols}")
        print(f"  Total Cells: {n_rows * n_cols:,}")
        eda_results['shape'] = (n_rows, n_cols)
        print()
        
        # 2. Duplicated Rows
        print("🔄 DUPLICATED ROWS")
        print("-" * 80)
        n_duplicates = self.df.duplicated().sum()
        pct_duplicates = (n_duplicates / n_rows * 100) if n_rows > 0 else 0
        print(f"  Total Duplicated Rows: {n_duplicates:,}")
        print(f"  Percentage: {pct_duplicates:.2f}%")
        eda_results['duplicates'] = n_duplicates
        print()
        
        # 3. Missing Values
        print("❌ MISSING VALUES")
        print("-" * 80)
        missing_data = self.df.isnull().sum()
        missing_pct = (missing_data / n_rows * 100).round(2)
        
        missing_info = pd.DataFrame({
            'Column': missing_data.index,
            'Missing_Count': missing_data.values,
            'Missing_%': missing_pct.values
        }).sort_values('Missing_%', ascending=False)
        
        print(missing_info.to_string(index=False))
        eda_results['missing_summary'] = missing_info
        print()
        
        # 4. Numeric Columns Statistics
        print("🔢 NUMERIC COLUMNS STATISTICS")
        print("-" * 80)
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        
        if numeric_cols:
            numeric_stats = self.df[numeric_cols].describe(
                percentiles=[0.25, 0.5, 0.75]
            ).T
            
            # Add skewness and kurtosis
            numeric_stats['skewness'] = self.df[numeric_cols].skew()
            numeric_stats['kurtosis'] = self.df[numeric_cols].kurtosis()
            
            print(numeric_stats.to_string())
            eda_results['numeric_stats'] = numeric_stats
        else:
            print("  No numeric columns found.")
        print()
        
        # 5. Categorical Columns Statistics
        print("📝 CATEGORICAL COLUMNS STATISTICS")
        print("-" * 80)
        categorical_cols = self.df.select_dtypes(include=['object']).columns.tolist()
        
        if categorical_cols:
            for col in categorical_cols:
                n_unique = self.df[col].nunique()
                top_value = self.df[col].mode().values[0] if len(self.df[col].mode()) > 0 else 'N/A'
                top_count = self.df[col].value_counts().iloc[0] if len(self.df[col].value_counts()) > 0 else 0
                top_pct = (top_count / n_rows * 100) if n_rows > 0 else 0
                
                print(f"  Column: {col}")
                print(f"    - Unique Values: {n_unique}")
                print(f"    - Top Value: {top_value}")
                print(f"    - Top Count: {top_count:,}")
                print(f"    - Top %: {top_pct:.2f}%")
                print()
        else:
            print("  No categorical columns found.")
        print()
        
        print("="*80)
        return eda_results

## 2.2 Load Raw Data

In [ ]:
# ============================================================================
# CELL 4: Load Raw Data from CSV
# ============================================================================

# Load the dataset
df_raw = pd.read_csv(DATA_INPUT_PATH, encoding='utf-8')

print(f"✓ Raw data loaded successfully: {df_raw.shape}")
print(f"\nColumn Names:")
for idx, col in enumerate(df_raw.columns, 1):
    print(f"  {idx:2d}. {col}")
print()

## 2.3 Run Comprehensive EDA on Raw Data

In [ ]:
# ============================================================================
# CELL 5: Run Comprehensive EDA Analysis
# ============================================================================

# Create analyzer instance and run EDA
analyzer_raw = DataAnalyzer(df_raw, name="Raw Data")
eda_results_raw = analyzer_raw.comprehensive_eda()

## 2.4 Visualize Missing Values Pattern

In [ ]:
# ============================================================================
# CELL 6: Visualize Missing Values Heatmap
# ============================================================================

# Create a visualization of missing values
fig, axes = plt.subplots(2, 1, figsize=(16, 10), dpi=100)

# 1. Missing Value Heatmap
missing_data = df_raw.isnull()
sns.heatmap(
    missing_data.iloc[:100, :],  # Show first 100 rows for readability
    cbar=True,
    yticklabels=False,
    cmap='YlOrRd',
    ax=axes[0]
)
axes[0].set_title('Missing Values Heatmap (First 100 Rows)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Columns')

# 2. Missing Value Percentage Bar Plot
missing_pct = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]  # Only show columns with missing values

if len(missing_pct) > 0:
    axes[1].barh(range(len(missing_pct)), missing_pct.values, color='coral')
    axes[1].set_yticks(range(len(missing_pct)))
    axes[1].set_yticklabels(missing_pct.index)
    axes[1].set_xlabel('Missing %', fontsize=12)
    axes[1].set_title('Missing Values Percentage by Column', fontsize=14, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'No Missing Values Found', ha='center', va='center', fontsize=12)
    axes[1].set_title('Missing Values Percentage by Column', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '00_missing_values_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Missing values visualization saved to data_output/00_missing_values_analysis.png")
print()

---

# 3. Define Custom Transformers (OOP/Modular Design) <a name="transformers"></a>

## 3.1 Base Abstract Transformer Class

In [ ]:
# ============================================================================
# CELL 7: Define Base Abstract Transformer Class
# ============================================================================

class BaseDataTransformer(BaseEstimator, TransformerMixin, ABC):
    """
    Abstract base class for custom data transformers.
    Follows scikit-learn Transformer pattern for pipeline compatibility.
    
    All subclasses must implement:
    - fit(X, y=None) -> returns self
    - transform(X) -> returns transformed X
    - validate_after_transform() -> raises AssertionError if validation fails
    """
    
    def fit(self, X: pd.DataFrame, y=None) -> 'BaseDataTransformer':
        """
        Fit the transformer (no-op for stateless transformers).
        
        Args:
            X (pd.DataFrame): Input data
            y: Target variable (ignored, for pipeline compatibility)
        
        Returns:
            self
        """
        return self
    
    @abstractmethod
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Apply transformation to the data.
        Must be implemented by subclasses.
        
        Args:
            X (pd.DataFrame): Input data
        
        Returns:
            pd.DataFrame: Transformed data
        """
        pass
    
    @abstractmethod
    def validate_after_transform(self, X: pd.DataFrame) -> None:
        """
        Perform data assertions to validate transformation.
        Must be implemented by subclasses.
        
        Args:
            X (pd.DataFrame): Transformed data
        
        Raises:
            AssertionError: If validation fails
        """
        pass
    
    def fit_transform(self, X: pd.DataFrame, y=None) -> pd.DataFrame:
        """
        Fit and transform in one step, with built-in validation.
        
        Args:
            X (pd.DataFrame): Input data
            y: Target variable (ignored)
        
        Returns:
            pd.DataFrame: Transformed data
        """
        X_transformed = self.fit(X, y).transform(X)
        self.validate_after_transform(X_transformed)
        return X_transformed

## 3.2 Drop Unnecessary Columns Transformer

In [ ]:
# ============================================================================
# CELL 8: DropColumnsTransformer - Remove Columns with High Missing Rate
# ============================================================================

class DropColumnsTransformer(BaseDataTransformer):
    """
    Drop specified columns from the dataset.
    
    This transformer removes columns with high missing value rates,
    as specified in the data cleaning requirements:
    - 'Hướng ban công'
    - 'Hướng nhà'
    """
    
    def __init__(self, columns_to_drop: List[str]):
        """
        Initialize the transformer.
        
        Args:
            columns_to_drop (List[str]): List of column names to drop
        """
        self.columns_to_drop = columns_to_drop
    
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Drop specified columns.
        
        Args:
            X (pd.DataFrame): Input dataframe
        
        Returns:
            pd.DataFrame: Dataframe with specified columns removed
        """
        X_copy = X.copy()
        cols_to_drop = [col for col in self.columns_to_drop if col in X_copy.columns]
        
        if cols_to_drop:
            logger.info(f"Dropping columns: {cols_to_drop}")
            X_copy.drop(columns=cols_to_drop, inplace=True)
        
        return X_copy
    
    def validate_after_transform(self, X: pd.DataFrame) -> None:
        """
        Validate that specified columns have been removed.
        
        Args:
            X (pd.DataFrame): Transformed dataframe
        
        Raises:
            AssertionError: If any specified column still exists
        """
        for col in self.columns_to_drop:
            assert col not in X.columns, f"Column '{col}' should have been dropped but still exists!"
        logger.info(f"✓ Assertion passed: All specified columns dropped successfully")

## 3.3 Text Normalization Transformer

In [ ]:
# ============================================================================
# CELL 9: TextNormalizationTransformer - Normalize Text Data
# ============================================================================

class TextNormalizationTransformer(BaseDataTransformer):
    """
    Normalize text/categorical columns with the following steps:
    1. Cast to string
    2. Unicode normalization (NFC)
    3. Strip whitespace
    4. Remove trailing periods
    5. Convert to lowercase
    6. Convert empty strings to np.nan
    """
    
    def __init__(self, columns_to_normalize: List[str]):
        """
        Initialize the transformer.
        
        Args:
            columns_to_normalize (List[str]): List of column names to normalize
        """
        self.columns_to_normalize = columns_to_normalize
    
    @staticmethod
    def normalize_text(text: Any) -> Optional[str]:
        """
        Normalize a single text value following the required steps.
        
        Normalization Pipeline:
        1. Cast to string
        2. Unicode normalization (NFC form)
        3. Strip leading/trailing whitespace
        4. Remove trailing periods
        5. Convert to lowercase
        6. If result is empty string, convert to np.nan
        
        Args:
            text (Any): Input value (may be string, number, or null)
        
        Returns:
            Optional[str]: Normalized text or np.nan
        """
        # Handle null values
        if pd.isna(text):
            return np.nan
        
        # Step 1: Cast to string
        text_str = str(text)
        
        # Step 2: Unicode normalization (NFC form)
        text_str = unicodedata.normalize('NFC', text_str)
        
        # Step 3: Strip whitespace from both ends
        text_str = text_str.strip()
        
        # Step 4: Remove trailing periods
        text_str = text_str.rstrip('.')
        
        # Step 5: Convert to lowercase
        text_str = text_str.lower()
        
        # Step 6: Convert empty strings to np.nan
        if text_str == '':
            return np.nan
        
        return text_str
    
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Apply text normalization to specified columns.
        
        Args:
            X (pd.DataFrame): Input dataframe
        
        Returns:
            pd.DataFrame: Dataframe with normalized text columns
        """
        X_copy = X.copy()
        
        cols_to_process = [col for col in self.columns_to_normalize if col in X_copy.columns]
        
        logger.info(f"Normalizing {len(cols_to_process)} text columns: {cols_to_process}")
        
        for col in cols_to_process:
            X_copy[col] = X_copy[col].apply(self.normalize_text)
        
        return X_copy
    
    def validate_after_transform(self, X: pd.DataFrame) -> None:
        """
        Validate text normalization results.
        
        Checks:
        - No values end with periods (\d.)
        - No uppercase letters in normalized columns
        - No leading/trailing whitespace in non-null values
        
        Args:
            X (pd.DataFrame): Transformed dataframe
        
        Raises:
            AssertionError: If any validation check fails
        """
        cols_to_check = [col for col in self.columns_to_normalize if col in X.columns]
        
        for col in cols_to_check:
            # Check 1: No periods at the end
            has_trailing_period = X[col].astype(str).str.endswith('.').sum()
            assert has_trailing_period == 0, f"Column '{col}' still has {has_trailing_period} values ending with periods!"
            
            # Check 2: No uppercase letters
            has_uppercase = X[col].dropna().astype(str).str.contains('[A-Z]').sum()
            assert has_uppercase == 0, f"Column '{col}' still has {has_uppercase} uppercase letters!"
            
            # Check 3: No leading/trailing whitespace in non-null values
            non_null_values = X[col].dropna()
            if len(non_null_values) > 0:
                has_whitespace = (non_null_values.astype(str) != non_null_values.astype(str).str.strip()).sum()
                assert has_whitespace == 0, f"Column '{col}' has {has_whitespace} values with leading/trailing whitespace!"
        
        logger.info(f"✓ Assertion passed: All text normalization checks successful")

---

# 4. Execute Data Cleaning Pipeline <a name="pipeline"></a>

## 4.1 Create scikit-learn Pipeline

In [ ]:
# ============================================================================
# CELL 10: Create Modular Pipeline with Custom Transformers
# ============================================================================

# Define columns to drop (high missing rate)
COLUMNS_TO_DROP = ['Hướng ban công', 'Hướng nhà']

# Define text columns to normalize
TEXT_COLUMNS_TO_NORMALIZE = [
    'Pháp lý',
    'Nội thất',
    'Loại đường vào',
    'Quan_Huyen',
    'Phuong_Xa',
    'Ten_Do_An',
    'QHPK_Chuc_Nang',
    'QHPK_Ma_QU',
    'QHPK_Ma_O_Pho',
    'Trang_Thai',
    'Đường vào'
]

print("="*80)
print("PIPELINE CONFIGURATION")
print("="*80)
print(f"\nColumns to Drop: {COLUMNS_TO_DROP}")
print(f"\nText Columns to Normalize ({len(TEXT_COLUMNS_TO_NORMALIZE)}):")
for col in TEXT_COLUMNS_TO_NORMALIZE:
    print(f"  - {col}")
print()

# Create the cleaning pipeline
cleaning_pipeline = Pipeline([
    ('drop_columns', DropColumnsTransformer(columns_to_drop=COLUMNS_TO_DROP)),
    ('normalize_text', TextNormalizationTransformer(columns_to_normalize=TEXT_COLUMNS_TO_NORMALIZE))
])

print("✓ Pipeline created successfully")
print(f"\nPipeline steps: {[step[0] for step in cleaning_pipeline.steps]}")
print()

## 4.2 Execute the Pipeline

In [ ]:
# ============================================================================
# CELL 11: Execute Data Cleaning Pipeline with Validation
# ============================================================================

print("\n" + "="*80)
print("EXECUTING DATA CLEANING PIPELINE")
print("="*80 + "\n")

try:
    # Apply the pipeline with full validation
    df_cleaned = cleaning_pipeline.fit_transform(df_raw)
    logger.info(f"✓ Pipeline executed successfully!")
    logger.info(f"  - Original shape: {df_raw.shape}")
    logger.info(f"  - Cleaned shape: {df_cleaned.shape}")
    logger.info(f"  - Rows removed: {df_raw.shape[0] - df_cleaned.shape[0]}")
    logger.info(f"  - Columns removed: {df_raw.shape[1] - df_cleaned.shape[1]}")
except Exception as e:
    logger.error(f"❌ Pipeline execution failed: {str(e)}")
    raise

print()

## 4.3 Run EDA on Cleaned Data

In [ ]:
# ============================================================================
# CELL 12: Comprehensive EDA on Cleaned Data
# ============================================================================

analyzer_cleaned = DataAnalyzer(df_cleaned, name="Cleaned Data")
eda_results_cleaned = analyzer_cleaned.comprehensive_eda()

## 4.4 Compare Raw vs Cleaned Data

In [ ]:
# ============================================================================
# CELL 13: Comparison Between Raw and Cleaned Data
# ============================================================================

print("\n" + "="*80)
print("BEFORE & AFTER COMPARISON")
print("="*80 + "\n")

comparison_data = {
    'Metric': [
        'Total Rows',
        'Total Columns',
        'Duplicated Rows',
        'Total Missing Values',
        'Avg Missing % per Column'
    ],
    'Raw Data': [
        f"{df_raw.shape[0]:,}",
        f"{df_raw.shape[1]}",
        f"{df_raw.duplicated().sum():,}",
        f"{df_raw.isnull().sum().sum():,}",
        f"{(df_raw.isnull().sum().sum() / (df_raw.shape[0] * df_raw.shape[1]) * 100):.2f}%"
    ],
    'Cleaned Data': [
        f"{df_cleaned.shape[0]:,}",
        f"{df_cleaned.shape[1]}",
        f"{df_cleaned.duplicated().sum():,}",
        f"{df_cleaned.isnull().sum().sum():,}",
        f"{(df_cleaned.isnull().sum().sum() / (df_cleaned.shape[0] * df_cleaned.shape[1]) * 100):.2f}%"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print()

---

# 5. Train/Test Split & Export Data <a name="export"></a>

## 5.1 Prepare Features and Target

In [ ]:
# ============================================================================
# CELL 14: Identify Target Variable and Prepare X, y
# ============================================================================

# Identify the target variable (last column is 'Trang_Thai')
TARGET_COLUMN = 'Trang_Thai'

print(f"📊 TARGET VARIABLE: {TARGET_COLUMN}")
print(f"\nTarget Value Distribution:")
print(df_cleaned[TARGET_COLUMN].value_counts())
print()

# Verify target exists
assert TARGET_COLUMN in df_cleaned.columns, f"Target column '{TARGET_COLUMN}' not found in cleaned data!"
print(f"✓ Target column verified")
print()

## 5.2 Train/Test Split

In [ ]:
# ============================================================================
# CELL 15: Perform Train/Test Split (80/20)
# ============================================================================

# Random seed for reproducibility
RANDOM_STATE = 42
TEST_SIZE = 0.2

# Prepare features and target
X = df_cleaned.drop(columns=[TARGET_COLUMN])
y = df_cleaned[TARGET_COLUMN]

print(f"Original dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Test size: {TEST_SIZE*100:.0f}%")
print()

# Perform train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y  # Ensure target distribution is preserved
)

print("✓ Train/Test split completed:")
print(f"  - Train set: {X_train.shape[0]} samples ({X_train.shape[0]/X.shape[0]*100:.1f}%)")
print(f"  - Test set: {X_test.shape[0]} samples ({X_test.shape[0]/X.shape[0]*100:.1f}%)")
print()

# Verify stratification
print("Train set target distribution:")
print(y_train.value_counts(normalize=True).round(4))
print()
print("Test set target distribution:")
print(y_test.value_counts(normalize=True).round(4))
print()

## 5.3 Merge X and y, Then Export

In [ ]:
# ============================================================================
# CELL 16: Merge X and y, Export to CSV
# ============================================================================

# Merge X and y back together for export
train_data = X_train.copy()
train_data[TARGET_COLUMN] = y_train.values

test_data = X_test.copy()
test_data[TARGET_COLUMN] = y_test.values

# Ensure target column is at the end (common convention)
train_data = train_data[[col for col in train_data.columns if col != TARGET_COLUMN] + [TARGET_COLUMN]]
test_data = test_data[[col for col in test_data.columns if col != TARGET_COLUMN] + [TARGET_COLUMN]]

print("="*80)
print("EXPORTING CLEANED DATA")
print("="*80 + "\n")

# Export train data
train_data.to_csv(TRAIN_OUTPUT_PATH, index=False, encoding='utf-8')
logger.info(f"✓ Train data exported: {TRAIN_OUTPUT_PATH}")
logger.info(f"  - Shape: {train_data.shape}")
logger.info(f"  - Size: {TRAIN_OUTPUT_PATH.stat().st_size / (1024*1024):.2f} MB")
print()

# Export test data
test_data.to_csv(TEST_OUTPUT_PATH, index=False, encoding='utf-8')
logger.info(f"✓ Test data exported: {TEST_OUTPUT_PATH}")
logger.info(f"  - Shape: {test_data.shape}")
logger.info(f"  - Size: {TEST_OUTPUT_PATH.stat().st_size / (1024*1024):.2f} MB")
print()

---

# 6. Final Data Assertions & Validation <a name="assertions"></a>

## 6.1 Comprehensive Data Validation

In [ ]:
# ============================================================================
# CELL 17: Final Data Assertions & Sanity Checks
# ============================================================================

class DataValidator:
    """
    Comprehensive data validator for post-cleaning assertions.
    """
    
    @staticmethod
    def validate_all(df_train: pd.DataFrame, df_test: pd.DataFrame, target_col: str) -> None:
        """
        Run all validation checks.
        
        Args:
            df_train (pd.DataFrame): Training dataset
            df_test (pd.DataFrame): Test dataset
            target_col (str): Name of target column
        
        Raises:
            AssertionError: If any validation check fails
        """
        print("\n" + "="*80)
        print("RUNNING FINAL DATA ASSERTIONS")
        print("="*80 + "\n")
        
        # 1. Check removed columns
        print("[1/8] Checking removed columns...")
        assert 'Hướng nhà' not in df_train.columns, "'Hướng nhà' should be removed!"
        assert 'Hướng ban công' not in df_train.columns, "'Hướng ban công' should be removed!"
        assert 'Hướng nhà' not in df_test.columns, "'Hướng nhà' should be removed!"
        assert 'Hướng ban công' not in df_test.columns, "'Hướng ban công' should be removed!"
        print("  ✓ Pass: Unnecessary columns removed")
        
        # 2. Check target column exists
        print("[2/8] Checking target column...")
        assert target_col in df_train.columns, f"Target column '{target_col}' missing in train set!"
        assert target_col in df_test.columns, f"Target column '{target_col}' missing in test set!"
        print(f"  ✓ Pass: Target column '{target_col}' exists")
        
        # 3. Check no trailing periods in text columns
        print("[3/8] Checking for trailing periods in text columns...")
        text_cols = df_train.select_dtypes(include=['object']).columns
        for col in text_cols:
            n_trailing_periods = df_train[col].astype(str).str.endswith('.').sum()
            assert n_trailing_periods == 0, f"Column '{col}' has {n_trailing_periods} trailing periods!"
        print("  ✓ Pass: No trailing periods in text columns")
        
        # 4. Check no uppercase in normalized columns
        print("[4/8] Checking for uppercase letters in normalized columns...")
        for col in text_cols:
            n_uppercase = df_train[col].dropna().astype(str).str.contains('[A-Z]').sum()
            assert n_uppercase == 0, f"Column '{col}' has {n_uppercase} uppercase letters!"
        print("  ✓ Pass: No uppercase letters in text columns")
        
        # 5. Check column alignment
        print("[5/8] Checking feature alignment...")
        train_cols = set(df_train.columns)
        test_cols = set(df_test.columns)
        assert train_cols == test_cols, "Train and test columns don't match!"
        print(f"  ✓ Pass: Train and test have identical columns ({len(train_cols)} columns)")
        
        # 6. Check no data leakage
        print("[6/8] Checking for potential data leakage...")
        train_indices = set(df_train.index)
        test_indices = set(df_test.index)
        overlap = train_indices & test_indices
        assert len(overlap) == 0, f"Found {len(overlap)} overlapping indices between train and test!"
        print("  ✓ Pass: No data leakage detected")
        
        # 7. Check stratification
        print("[7/8] Checking target stratification...")
        train_dist = df_train[target_col].value_counts(normalize=True).sort_index()
        test_dist = df_test[target_col].value_counts(normalize=True).sort_index()
        
        # Allow up to 5% difference in distribution
        max_diff = (train_dist - test_dist).abs().max()
        assert max_diff < 0.05, f"Poor stratification: max distribution difference {max_diff:.3f}!"
        print(f"  ✓ Pass: Target stratification successful (max diff: {max_diff:.3f})")
        
        # 8. Check file exports
        print("[8/8] Checking file exports...")
        assert TRAIN_OUTPUT_PATH.exists(), f"Train file not exported: {TRAIN_OUTPUT_PATH}"
        assert TEST_OUTPUT_PATH.exists(), f"Test file not exported: {TEST_OUTPUT_PATH}"
        print(f"  ✓ Pass: Both output files exist")
        
        print("\n" + "="*80)
        print("✓ ALL ASSERTIONS PASSED - DATA CLEANING SUCCESSFUL")
        print("="*80)

# Run validation
DataValidator.validate_all(train_data, test_data, TARGET_COLUMN)

## 6.2 Final Summary Report

In [ ]:
# ============================================================================
# CELL 18: Generate Final Summary Report
# ============================================================================

print("\n" + "="*80)
print("📋 FINAL SUMMARY REPORT")
print("="*80 + "\n")

print("1️⃣  DATA PROCESSING WORKFLOW:")
print("-" * 80)
print("  Step 1: Load & EDA - Analyzed raw data structure and quality")
print("  Step 2: Drop Columns - Removed 'Hướng nhà' & 'Hướng ban công' (high missing %)")
print("  Step 3: Text Normalization - Applied 6-step text cleaning to categorical columns")
print("  Step 4: Train/Test Split - 80/20 split with stratification")
print("  Step 5: Export - Saved cleaned data to CSV files")
print()

print("2️⃣  DATASET STATISTICS:")
print("-" * 80)
print(f"  Raw Data:")
print(f"    - Rows: {df_raw.shape[0]:,}")
print(f"    - Columns: {df_raw.shape[1]}")
print(f"    - Missing Values: {df_raw.isnull().sum().sum():,}")
print()
print(f"  Cleaned Data:")
print(f"    - Rows: {df_cleaned.shape[0]:,}")
print(f"    - Columns: {df_cleaned.shape[1]}")
print(f"    - Missing Values: {df_cleaned.isnull().sum().sum():,}")
print(f"    - Improvement: {df_raw.shape[1] - df_cleaned.shape[1]} fewer columns")
print()

print("3️⃣  TRAIN/TEST SPLIT:")
print("-" * 80)
print(f"  Train Set: {train_data.shape[0]:,} samples ({train_data.shape[0]/len(df_cleaned)*100:.1f}%)")
print(f"  Test Set: {test_data.shape[0]:,} samples ({test_data.shape[0]/len(df_cleaned)*100:.1f}%)")
print(f"  Features: {train_data.shape[1] - 1}")
print(f"  Target: {TARGET_COLUMN}")
print()

print("4️⃣  OUTPUT FILES:")
print("-" * 80)
print(f"  📁 Directory: {OUTPUT_DIR}")
print(f"  📄 Train Data: {TRAIN_OUTPUT_PATH.name} ({TRAIN_OUTPUT_PATH.stat().st_size / (1024) if TRAIN_OUTPUT_PATH.exists() else 0 / 1024:.1f} KB)")
print(f"  📄 Test Data: {TEST_OUTPUT_PATH.name} ({TEST_OUTPUT_PATH.stat().st_size / (1024) if TEST_OUTPUT_PATH.exists() else 0 / 1024:.1f} KB)")
print()

print("5️⃣  ARCHITECTURE & CODE QUALITY:")
print("-" * 80)
print("  ✓ Modular OOP: BaseDataTransformer + scikit-learn Pipeline")
print("  ✓ Data Assertions: 8 validation checks after cleaning")
print("  ✓ Reproducibility: Relative paths, RANDOM_STATE=42, no hardcoded values")
print("  ✓ Clean Code: PEP 8, Type Hints, Docstrings, Logging")
print("  ✓ End-to-End: Complete pipeline from raw data to split datasets")
print()

print("="*80)
print("✨ DATA CLEANING PIPELINE COMPLETED SUCCESSFULLY ✨")
print("="*80)

## 6.3 Display Sample Output

In [ ]:
# ============================================================================
# CELL 19: Display Sample Output Data
# ============================================================================

print("\n" + "="*80)
print("SAMPLE CLEANED DATA")
print("="*80 + "\n")

print("Train Data (First 5 rows):")
print("-" * 80)
print(train_data.head(5).to_string())
print()

print("Column Data Types:")
print("-" * 80)
print(train_data.dtypes)
print()

print("Missing Values in Cleaned Data:")
print("-" * 80)
missing_clean = train_data.isnull().sum()
missing_clean = missing_clean[missing_clean > 0]
if len(missing_clean) > 0:
    print(missing_clean)
else:
    print("  No missing values found in all numeric columns!")
print()

print("="*80)